In [1]:
import pandas as pd
import json
from pathlib import Path





In [2]:
INPUT_FILE = r"D:\feature 4\experence3\stackoverflow.com-Posts.jsonl"
OUTPUT_FILE = r"D:\feature 4\experence3\data.csv"
CHUNK_SIZE = 200_000
TARGET_SINGLE_TOTAL = 1000_000
TARGET_DOUBLE_TOTAL = 701_000

In [3]:
se_tags = {
    "python",
    "javascript",
    "typescript",
    "java",
    "c#",
    "sql",
    "django",
    "fastapi",
    "node.js",
    "spring-boot",
    "asp.net-core",
    "reactjs",
    "rest",
    "authentication",
    "jwt",
    "oauth-2.0",
    "debugging",
    "logging",
    "postgresql",
    "redis",
}

net_tags = {
    "networking",
    "linux",
    "docker",
    "kubernetes",
    "nginx",
    "apache",
    "http",
    "dns",
    "ssl",
    "tls",
    "cors",
    "ssh",
    "tcp",
    "proxy",
    "load-balancing",
    "firewall",
    "ingress",
    "server",
    "bash",
    "azure",
}

ai_tags = {
    "machine-learning",
    "deep-learning",
    "tensorflow",
    "pytorch",
    "scikit-learn",
    "pandas",
    "numpy",
    "opencv",
    "computer-vision",
    "nlp",
    "transformers",
    "llm",
    "rag",
    "embeddings",
    "vector-database",
    "model-serving",
    "gpu",
    "cuda",
    "inference",
    "mlops",
}

In [4]:

allowed_tags = se_tags | net_tags | ai_tags

first_chunk = True
single_count = 0
double_count = 0
for chunk in pd.read_json(INPUT_FILE, lines=True, chunksize=CHUNK_SIZE):
    chunk["text"] = chunk["texts"].apply(
        lambda x: " ".join(map(str, x)) if isinstance(x, list) else ""
    )
    chunk["tags"] = chunk["tags"].apply(
        lambda x: " ".join([t for t in map(str, x) if t in allowed_tags]) if isinstance(x, list) else ""
    )

    chunk = chunk[["text", "tags"]].copy()

    chunk["text"] = chunk["text"].fillna("").astype(str).str.strip()
    chunk["tags"] = chunk["tags"].fillna("").astype(str).str.strip()

    chunk = chunk[(chunk["text"] != "") & (chunk["tags"] != "")].copy()
    chunk["num_tags"] = chunk["tags"].apply(lambda x: len(x.split()))
 # 1-tag
    single_df = chunk[chunk["num_tags"] == 1].copy()

    # 2-tags
    double_df = chunk[chunk["num_tags"] == 2].copy()

    # 3+ tags
    multi3_df = chunk[chunk["num_tags"] >= 3].copy()

    remaining_single = TARGET_SINGLE_TOTAL - single_count
    if remaining_single > 0 and len(single_df) > 0:
        single_df = single_df.sample(
            n=min(remaining_single, len(single_df)),
            random_state=42
        )
        single_count += len(single_df)
    else:
        single_df = single_df.iloc[0:0]

    remaining_double = TARGET_DOUBLE_TOTAL - double_count
    if remaining_double > 0 and len(double_df) > 0:
        double_df = double_df.sample(
            n=min(remaining_double, len(double_df)),
            random_state=42
        )
        double_count += len(double_df)
    else:
        double_df = double_df.iloc[0:0]

    # الدمج النهائي
    selected_df = pd.concat([ double_df, multi3_df,single_df], ignore_index=True)

    if len(selected_df) > 0:
        selected_df.to_csv(
            OUTPUT_FILE,
            mode="a",
            index=False,
            header=first_chunk,
            encoding="utf-8-sig"
        )
        first_chunk = False

print("Done")


Done


In [8]:
import pandas as pd

df = pd.read_csv(r"D:\feature 4\experence3\data.csv")

df["num_tags"] = df["tags"].fillna("").apply(lambda x: len(str(x).split()))

print(df["num_tags"].value_counts().sort_index())


num_tags
1    1000000
2     701000
3     112707
4       7949
5        304
Name: count, dtype: int64


In [6]:
print(df.head(10))

                                                text             tags  \
0  Loop through elements of list in a pandas data...    python pandas   
1  SSL Error when running pip search in python 2....       python ssl   
2  How do I clear errno in C#? How do I clear err...         c# linux   
3  Segmentation fault as soon the binary launch H...  linux debugging   
4  Changing data in jsp using ajax I have jsp pag...  javascript java   
5  Python: pandas dataframe comparison of rows wi...    python pandas   
6  java batch select or multiple selects in paral...         java sql   
7  Render a continually changing numpy array to s...    python opencv   
8  why "django.forms.widgets has no attribute Rad...    python django   
9  Java Screen Scrape using sockets? I am trying ...  java networking   

   num_tags  
0         2  
1         2  
2         2  
3         2  
4         2  
5         2  
6         2  
7         2  
8         2  
9         2  


In [5]:
import re
import html

NORMALIZATION_MAP = {
    "c sharp": "c#",
    "c-sharp": "c#",
    "c plus plus": "c++",
    "cpp": "c++",
    "js": "javascript",
    "nodejs": "node.js",
    "asp net": "asp.net",
    "ms sql": "sql server",
    "mssql": "sql server",
    "postgres": "postgresql",
}

def normalize_terms(text):
    for src, tgt in NORMALIZATION_MAP.items():
        pattern = r"\b" + re.escape(src) + r"\b"
        text = re.sub(pattern, tgt, text)
    return text

def clean_technical_text(text):
    text = str(text)

    # فك HTML
    text = html.unescape(text)
# code/pre blocks
    text = re.sub(r"<code>.*?</code>", " CODEBLOCK ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<pre>.*?</pre>", " CODEBLOCK ", text, flags=re.DOTALL | re.IGNORECASE)
# حذف HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # lowercase
    text = text.lower()

    # normalization (مهم جداً)
    text = normalize_terms(text)

   # replace urls / emails
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " EMAIL ", text)
    

    # الحفاظ على الرموز التقنية
    text = re.sub(r"[^a-z0-9\s\#\+\.\-_/]", " ", text)

    # تنظيف المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [9]:
import pandas as pd

df["text"] = df["text"].fillna("").astype(str).apply(clean_technical_text)
# حذف أي صف نصه صار فارغ
df = df[df["text"].str.strip() != ""].copy()
df = df.drop("num_tags", axis=1)
df.to_csv(r"D:\feature 4\experence3\new_data.csv", index=False, encoding="utf-8-sig")



In [12]:
df["num_tag"] = df["tags"].fillna("").apply(lambda x: len(str(x).split()))

print(df["num_tag"].value_counts().sort_index())

num_tag
3    112707
4      7949
5       304
Name: count, dtype: int64


In [2]:
import pandas as pd

df2 = pd.read_csv(r"D:\feature 4\experence3\merged_text_tags_final_balanced.csv")
df1 = pd.read_csv(r"D:\feature 4\experence3\new_data.csv")



In [11]:
df2["num_tags"] = df2["Tags"].fillna("").apply(lambda x: len(str(x).split()))

# توزيع عدد التاغات: 1 tag, 2 tags, 3 tags ...
print(df2["num_tags"].value_counts().sort_index())

num_tags
1    1000000
2     213943
3      18578
4       1245
5         51
Name: count, dtype: int64


In [6]:
import pandas as pd

df2 = pd.read_csv(r"D:\feature 4\experence3\merged_text_tags_final_balanced.csv")
df1 = pd.read_csv(r"D:\feature 4\experence3\new_data.csv")

# إعادة تسمية العمود Tags إلى tags في df2
df2 = df2.rename(columns={"Tags": "tags"})

df2["num_tags"] = df2["tags"].fillna("").astype(str).apply(lambda x: len(x.split()))
df2 = df2[df2["num_tags"] >= 3][["text", "tags"]].copy()

df1["num_tags"] = df1["tags"].fillna("").astype(str).apply(lambda x: len(x.split()))
df1 = df1[["text", "tags"]].copy()

final_df = pd.concat([df2, df1], ignore_index=True)

final_df["text"] = final_df["text"].fillna("").astype(str).str.strip()
final_df["tags"] = final_df["tags"].fillna("").astype(str).str.strip()

final_df = final_df[(final_df["text"] != "") & (final_df["tags"] != "")].copy()

final_df.to_csv(r"D:\feature 4\experence3\merged_all_datasets.csv", index=False, encoding="utf-8-sig")

In [7]:
final_df["num_tags"] = final_df["tags"].fillna("").apply(lambda x: len(str(x).split()))

print(final_df["num_tags"].value_counts().sort_index())

num_tags
1    1000000
2     701000
3     131285
4       9194
5        355
Name: count, dtype: int64
